# Семинар 2. PyTorch: от тензоров к обучению

По Chapter 1 книги Daniel Voigt Godoy. Код и порядок примеров сохранены;
адаптация курса добавляет MPS, переносимый запуск и заключительные задания.

## Содержание

1. [Знакомая задача и данные](#section-1)
2. [Шаги градиентного спуска](#section-2)
3. [Линейная регрессия в NumPy](#section-3)
4. [Тензоры, устройства и параметры PyTorch](#section-4)
5. [Autograd](#section-5)
6. [Динамический вычислительный граф](#section-6)
7. [Оптимизатор](#section-7)
8. [Функция потерь](#section-8)
9. [Модель](#section-9)
10. [Собираем всё вместе](#section-10)
11. [Задания](#section-11)

In [ ]:
from pathlib import Path
from runpy import run_path

# При согласованной проверке Colab замените main на SHA коммита.
course_revision = 'main'
try:
    import google.colab
except ModuleNotFoundError:
    repository_root = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / 'config.py').is_file() and (p / 'plots' / 'seminar02.py').is_file()), None)
    if repository_root is None:
        raise RuntimeError('Откройте notebook из локальной копии 2026_ML3.')
else:
    repository_root = Path.cwd().resolve()
    from urllib.request import urlopen
    url = f'https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/{course_revision}/config.py'
    with urlopen(url, timeout=30) as response:
        config_source = response.read()
    compile(config_source, 'config.py', 'exec')
    (repository_root / 'config.py').write_bytes(config_source)
course_config = run_path(str(repository_root / 'config.py'))
seminar_plots = course_config['config_seminar02'](branch=course_revision)
select_device = course_config['select_device']

from tempfile import TemporaryDirectory
if 'lesson_workspace' in globals():
    lesson_workspace.cleanup()
lesson_workspace = TemporaryDirectory(prefix='ml3-seminar02-')
lesson_dir = Path(lesson_workspace.name)
for folder in ('data_preparation', 'model_configuration', 'model_training'):
    (lesson_dir / folder).mkdir()

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

import torch
import torch.optim as optim
import torch.nn as nn
from torchviz import make_dot

# Это необходимо для визуализации графиков в этом уроке
figure1, figure3 = seminar_plots.figure1, seminar_plots.figure3

seminar_plots.configure_plots()
torch.manual_seed(42)

<a id="section-1"></a>

## 1. Знакомая задача и данные

$$
\Large y = b + w x + \epsilon
$$

### Генерация данных

#### Генерация синтетических данных

In [ ]:
true_b = 1
true_w = 2
N = 100

# Генерация данных
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w * x + epsilon

In [ ]:
# Перетасовывает индексы
idx = np.arange(N)
np.random.shuffle(idx)

# Использует первые 80 случайных индексов для обучения
train_idx = idx[:int(N*.8)]
# Использует оставшиеся индексы для валидации
val_idx = idx[int(N*.8):]

# Генерирует множества обучения и валидации
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

In [ ]:
figure1(x_train, y_train, x_val, y_val);

<a id="section-2"></a>

## 2. Шаги градиентного спуска


### Шаг 0: Случайная инициализация

In [ ]:
# Шаг 0 - Инициализирует параметры "b" и "w" случайным образом
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

### Шаг 1: Вычисление предсказаний модели

In [ ]:
# Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
yhat = b + w * x_train

### Шаг 2: Вычисление потерь

In [ ]:
# Шаг 2 - Вычисляет потери
# Мы используем все точки данных, таким образом это ПАКЕТНЫЙ градиентный
# спуск. Насколько ошибочная наша модель? Это определяется значением error!
error = (yhat - y_train)

# Это регрессия, поэтому рассчитывается средняя квадратичная ошибка (MSE)
loss = (error ** 2).mean()

print(loss)

### Шаг 3: Вычисление градиентов

In [ ]:
# Шаг 3 - вычисляет градиенты для параметров "b" и "w"
b_grad = 2 * error.mean()
w_grad = 2 * (x_train * error).mean()
print(b_grad, w_grad)

### Шаг 4: Обновление параметров

In [ ]:
# Устанавливает скорость обучения - это параметр "eta" - греческая буква похожая на "n"
lr = 0.1
print(b, w)

# Шаг 4 - Обновляет параметры, используя градиенты и
# скорость обучения
b = b - lr * b_grad
w = w - lr * w_grad

print(b, w)

### Шаг 5: Повторить!

In [ ]:
# Возвращаемся к шагу 1 и наблюдаем каким образом меняются параметры "b" и "w"

<a id="section-3"></a>

## 3. Линейная регрессия в NumPy


In [ ]:
# Шаг 0 - Случайным образом инициализирует параметры "b" и "w"
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)

print(b, w)

# Устанавливает скорость обучения - это параметр "eta" - греческая буква похожая на "n"
lr = 0.1
# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
    yhat = b + w * x_train
    
    # Шаг 2 - Вычисление потерь
    # Мы используем все точки данных, таким образом это ПАКЕТНЫЙ градиентный
    # спуск. Насколько ошибочная наша модель? Это определяется значением error!
    error = (yhat - y_train)
    # Это регрессия, таким образом вычисляет среднюю квадратичную ошибку (MSE)
    loss = (error ** 2).mean()
    
    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    b_grad = 2 * error.mean()
    w_grad = 2 * (x_train * error).mean()
    
    # Шаг 4 - Обновляет параметры используя градиенты и
    # скорость обучения
    b = b - lr * b_grad
    w = w - lr * w_grad
    
print(b, w)

In [ ]:
# Проверка на правильность: получим ли мы те же результаты, что и
# при градиентном спуске?
linr = LinearRegression()
linr.fit(x_train, y_train)
print(linr.intercept_, linr.coef_[0])

In [ ]:
fig = figure3(x_train, y_train)

<a id="section-4"></a>

## 4. Тензоры, устройства и параметры PyTorch


### Тензор

In [ ]:
import numpy as np

scalar_np = np.array(3.14159)
vector_np = np.array([1, 2, 3])
matrix_np = np.ones((2, 3), dtype=np.float32)
tensor_np = np.random.randn(2, 3, 4).astype(np.float32)
print(scalar_np)
print(vector_np)
print(matrix_np)
print(tensor_np)

In [ ]:
scalar = torch.tensor(3.14159)
vector = torch.tensor([1, 2, 3])
matrix = torch.ones((2, 3), dtype=torch.float)
tensor = torch.randn((2, 3, 4), dtype=torch.float)

print(scalar)
print(vector)
print(matrix)
print(tensor)

In [ ]:
print(tensor.size(), tensor.shape)

In [ ]:
print(scalar.size(), scalar.shape)

In [ ]:
# Мы получаем тензор с другой мерностью, но это всё ещё
# другое представление тех же данных
same_matrix = matrix.view(1, 6)
# Если мы изменим один из его элементов
same_matrix[0, 1] = 2.
# Меняются обе переменные: matrix и same_matrix
print(matrix)
print(same_matrix)

In [ ]:
# Мы можем использовать метод "new_tensor", чтобы
# ДЕЙСТВИТЕЛЬНО скопировать его в новый тензор
different_matrix = matrix.new_tensor(matrix.view(1, 6))
# Теперь, если мы изменим один из его элементов...
different_matrix[0, 1] = 3.
# Исходный тензор (матрица) остаётся нетронутым!
# Здесь ожидается предупреждение PyTorch
# предлагающее вместо этого использовать метод «clone»!
print(matrix)
print(different_matrix)

In [ ]:
# Давайте последуем предложению PyTorch и воспользуемся методом «clone»
another_matrix = matrix.view(1, 6).clone().detach()
# И снова, если мы изменим один из его элементов
another_matrix[0, 1] = 4.
# Исходный тензор (матрица) останется нетронутым!
print(matrix)
print(another_matrix)

### Загрузка данных и устройства

In [ ]:
x_train_tensor = torch.as_tensor(x_train)
x_train.dtype, x_train_tensor.dtype

In [ ]:
float_tensor = x_train_tensor.float()
float_tensor.dtype

In [ ]:
dummy_array = np.array([1, 2, 3])
dummy_tensor = torch.as_tensor(dummy_array)
# Изменяет массив Numpy
dummy_array[1] = 0
# Тензор также меняется...
dummy_tensor

In [ ]:
dummy_tensor.numpy()

#### Определение вашего устройства

In [ ]:
device = select_device()
print(f'Используемое устройство: {device}')

In [ ]:
n_cudas = torch.cuda.device_count()
for i in range(n_cudas):
    print(torch.cuda.get_device_name(i))

In [ ]:
gpu_tensor = torch.as_tensor(x_train).float().to(device)
gpu_tensor[0]

In [ ]:
device = select_device()

# Наши данные находятся в массивах Numpy, но нам нужно преобразовать их
# в тензоры PyTorch, а затем отправить их на
# выбранное устройство
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

In [ ]:
# Здесь мы видим разницу — обратите внимание, что .type() более
# полезен, поскольку он также сообщает нам, ГДЕ находится тензор (устройство)
print(type(x_train), type(x_train_tensor), x_train_tensor.type())

In [ ]:
# NumPy работает с CPU-памятью; на ускорителе нужен явный перенос.
try:
    back_to_numpy = x_train_tensor.numpy()
except (TypeError, RuntimeError):
    print('Для тензора на ускорителе сначала нужен .cpu().')
else:
    print('Тензор уже на CPU: numpy() доступен без переноса.')

In [ ]:
back_to_numpy = x_train_tensor.cpu().numpy()

### Создание параметров

In [ ]:
# ПЕРВЫЙ
# Инициализирует параметры "b" и "w" случайным образом, ПОЧТИ также как
# мы делали в Numpy, поскольку мы хотим применить градиентный спуск к
# этим параметрам, нам нужно установить REQUIRES_GRAD=TRUE
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float)
w = torch.randn(1, requires_grad=True, dtype=torch.float)
print(b, w)

In [ ]:
# ВТОРОЙ
# А что если мы хотим выполнить запуска на GPU? Мы можем просто
# отправить их на устройство, верно?
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float).to(device)
w = torch.randn(1, requires_grad=True, dtype=torch.float).to(device)
print(b, w)
# Если .to(device) копирует тензор на другое устройство, результат не лист.
# На CPU без смены dtype/device копии нет: тензор остаётся листом.
print('Листовые тензоры:', b.is_leaf, w.is_leaf)

In [ ]:
# ТРЕТИЙ
# Мы можем либо создать обычные тензоры и отправить их на
# устройство (как мы делали с нашими данными)
torch.manual_seed(42)
b = torch.randn(1, dtype=torch.float).to(device)
w = torch.randn(1, dtype=torch.float).to(device)
# а ЗАТЕМ сконфигурировать их как требующих градиентов...
b.requires_grad_()
w.requires_grad_()
print(b, w)

In [ ]:
# КОНЕЧНЫЙ
# Мы можем указать устройство в момент создания
# РЕКОМЕНДУЕТСЯ!

# Шаг 0 - Инициализирует параметры "b" и "w" случайным образом
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
print(b, w)

<a id="section-5"></a>

## 5. Autograd


### backward

Перед запуском: как пройти от среднего квадрата ошибки к каждому параметру?
Начните с производной потери по самой себе; по ребру умножайте на локальную
производную, вклады разных путей к одной переменной складывайте.

In [ ]:
# Шаг 1 - Вычисляет предсказанные выходные значения нашей модели - прямой проход
yhat = b + w * x_train_tensor

# Шаг 2 - Вычисляет потери
# Мы используем ВСЕ точки данных, таким образом это ПАКЕТНЫЙ градиентный
# спуск. Насколько ошибочная наша модель? Это определяется значением error!
error = (yhat - y_train_tensor)
# Это регрессия, поэтому вычисляется средняя квадратичная ошибка (MSE)
loss = (error ** 2).mean()

#  3 - Вычисляет градиенты для обоих параметров "b" и "w"
# Больше никаких ручных вычислений градиентов!
# b_grad = 2 * error.mean()
# w_grad = 2 * (x_tensor * error).mean()
loss.backward()

In [ ]:
print(error.requires_grad, yhat.requires_grad, \
      b.requires_grad, w.requires_grad)
print(y_train_tensor.requires_grad, x_train_tensor.requires_grad)

### grad

`None` означает отсутствие сохранённого градиента, а не обязательно нулевую
математическую производную. Для промежуточного тензора сохранение `.grad`
включается через `retain_grad()`.

In [ ]:
print(b.grad, w.grad)

In [ ]:
# Повторите forward/backward из раздела backward, затем вывод b.grad и w.grad.
# Градиенты накопятся; повтор одного print ничего не пересчитывает.

### zero_

Повторные forward/backward накапливают числа в `.grad`.
`retain_graph=True` сохраняет данные конкретного графа для повторного backward;
он не включает накопление и не вычисляет вторую производную.

In [ ]:
# Очищаем накопленные градиенты, не удаляя вычислительный граф.
# В полном цикле этот код будет после обновления параметров.
b.grad.zero_(), w.grad.zero_()

### Обновление параметров

In [ ]:
# Устанавливает скорость обучения - это параметра "eta", греческая буква похожая на "n"
lr = 0.1

# Шаг 0 - Случайным образом инициализирует параметры "b" и "w"
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
    yhat = b + w * x_train_tensor
    
    # Шаг 2 - Вычисляет потери
    # Мы используем ВСЕ точки данных, поэтому это ПАКЕТНЫЙ градиентный
    # спуск. Насколько ошибочная наша модель? Это определяется значением error!
    error = (yhat - y_train_tensor)
    # Это регрессия, поэтому вычисляет среднюю квадратичную ошибку (MSE)
    loss = (error ** 2).mean()

    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    # Больше никаких ручных вычислений градиентов!
    # b_grad = 2 * error.mean()
    # w_grad = 2 * (x_tensor * error).mean()   
    # Мы просто говорим PyTorch выполнить ОБРАТНЫЙ ПРОХОД
    # от указанного значения loss!
    loss.backward()
    
    # Шаг 4 - Обновление параметров с помощью градиентов и
    # скорости обучения. Но не так быстро...
    # ПЕРВАЯ ПОПЫТКА - просто используем тот же самый код, что и раньше
    # AttributeError: 'NoneType' object has no attribute 'zero_'
    # b = b - lr * b.grad
    # w = w - lr * w.grad
    # print(b)

    # ВТОРАЯ ПОПЫТКА - используем присваивание в терминах Python по месту
    # RuntimeError: a leaf Variable that requires grad
    # has been used in an in-place operation.
    # b -= lr * b.grad
    # w -= lr * w.grad        
    
    # ТРЕТЬЯ ПОПЫТКА - NO_GRAD для решения!
    # Нам нужно использовать NO_GRAD, чтобы развязать обновление с
    # вычислением градиента. Зачем это нужно? Всё упирается
    # в ДИНАМИЧЕСКИЙ ГРАФ, который использует PyTorch...
    with torch.no_grad():
        b -= lr * b.grad
        w -= lr * w.grad
    
    # Градиенты накапливаются в .grad; после шага очищаем эти поля.
    b.grad.zero_()
    w.grad.zero_()

print(b, w)

### no_grad

In [ ]:
# Это то, что мы использовали в ТРЕТЬЕЙ ПОПЫТКЕ...

<a id="section-6"></a>

## 6. Динамический вычислительный граф


Показан граф операций, реально выполненных при данном forward.
При следующем forward строится новый граф.

In [ ]:
# Шаг 0 - Случайным образом инициализирует параметры "b" и "w"
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
yhat = b + w * x_train_tensor

# Шаг 2 - Вычисляет потери
# Мы используем ВСЕ точки данных, поэтому это ПАКЕТНЫЙ градиентный
# спуск. Насколько неверная наша модель? Это определяется значением параметра error!
error = (yhat - y_train_tensor)
# Это регрессия, потому вычисляет среднюю квадратичную ошибку (MSE)
loss = (error ** 2).mean()

# Мы можем попробовать построить график для любой переменной Python:
# yhat, error, loss...
graph = make_dot(yhat)
graph.graph_attr.update(size='8,4')
graph

In [ ]:
b_nograd = torch.randn(1, requires_grad=False, \
                       dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

yhat = b_nograd + w * x_train_tensor

graph = make_dot(yhat)
graph.graph_attr.update(size='8,4')
graph

In [ ]:
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

yhat = b + w * x_train_tensor
error = yhat - y_train_tensor
loss = (error ** 2).mean()

# Это не имеет смысла!!
error2 = torch.zeros_like(error)
if loss > 0:
    yhat2 = w * x_train_tensor
    error2 = yhat2 - y_train_tensor
    
# И это тоже :-)
loss += error2.mean()

graph = make_dot(loss)
graph.graph_attr.update(size='8,5.5')
graph

<a id="section-7"></a>

## 7. Оптимизатор


### step / zero_grad

In [ ]:
# Определяет оптимизатор вида SGD для обновления параметров
optimizer = optim.SGD([b, w], lr=lr)

In [ ]:
# Устанавливает скорость обучения - это "eta" - греческая буква похожая на n
lr = 0.1

# Шаг 0 - Инициализирует параметры "b" и "w" случайным образом
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Определяет оптимизатор вида SGD для обновления параметров
optimizer = optim.SGD([b, w], lr=lr)

# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
    yhat = b + w * x_train_tensor
    
    # Шаг 2 - Вычисляет потери
    # Мы используем ВСЕ точки данных, поэтому это ПАКЕТНЫЙ градиентный
    # спуск. Насколько неверная наша модель? Это определяет величиной error!
    error = (yhat - y_train_tensor)
    # Это регрессия, поэтому вычисляет среднюю квадратичную ошибку (MSE)
    loss = (error ** 2).mean()

    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    loss.backward()
    
    # Шаг 4 - Обновляет параметры используя градиенты и
    # скорость обучения. Больше никаких ручных обновлений!
    # with torch.no_grad():
    #     b -= lr * b.grad
    #     w -= lr * w.grad
    optimizer.step()
    
    # Оптимизатор очищает накопленные градиенты параметров.
    # b.grad.zero_()
    # w.grad.zero_()
    optimizer.zero_grad()
    
print(b, w)

<a id="section-8"></a>

## 8. Функция потерь


In [ ]:
# Определяет функцию потерь MSE
loss_fn = nn.MSELoss(reduction='mean')
loss_fn

In [ ]:
# Это случайный пример для иллюстрации функции потерь
predictions = torch.tensor([0.5, 1.0])
labels = torch.tensor([2.0, 1.3])
loss_fn(predictions, labels)

In [ ]:
# Устанавливает скорость обучения - это "eta" — греческая буква
# похожая на n
lr = 0.1

# Шаг 0 - Случайным образом инициализирует параметры "b" и "w"
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)

# Определяет оптимизатор вида SGD для обновления параметров
optimizer = optim.SGD([b, w], lr=lr)

# Определяет функцию потерь как MSE
loss_fn = nn.MSELoss(reduction='mean')

# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
    yhat = b + w * x_train_tensor
    
    # Шаг 2 - Вычисляет потери
    # Больше никакой ручной работы с потерями!
    # error = (yhat - y_train_tensor)
    # loss = (error ** 2).mean()
    loss = loss_fn(yhat, y_train_tensor)

    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    loss.backward()
    
    # Шаг 4 - Обновляет параметры используя градиенты и
    # скорость обучения
    optimizer.step()
    optimizer.zero_grad()
    
print(b, w)

In [ ]:
loss

In [ ]:
# У loss отслеживается градиент: одного переноса на CPU недостаточно.
try:
    loss.cpu().numpy()
except RuntimeError:
    print('Перед преобразованием loss в NumPy нужен detach().')
else:
    raise AssertionError('Ожидалась ошибка для тензора с requires_grad=True')

In [ ]:
loss.detach().cpu().numpy()

In [ ]:
print(loss.item(), loss.tolist())

<a id="section-9"></a>

## 9. Модель


In [ ]:
class ManualLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        # Чтобы сделать "b" и "w" настоящими параметрами модели,
        # необходимо обернуть их с помощью nn.Parameter
        self.b = nn.Parameter(torch.randn(1,
                                          requires_grad=True, 
                                          dtype=torch.float))
        self.w = nn.Parameter(torch.randn(1, 
                                          requires_grad=True,
                                          dtype=torch.float))
        
    def forward(self, x):
        # Вычисляет выходные значения / предсказания
        return self.b + self.w * x

### Параметры

In [ ]:
torch.manual_seed(42)
# Создаёт "пустой" экземпляр для нашей модели ManualLinearRegression
dummy = ManualLinearRegression()
list(dummy.parameters())

### state_dict

In [ ]:
dummy.state_dict()

In [ ]:
optimizer.state_dict()

### Устройство

In [ ]:
torch.manual_seed(42)
# Создаёт "пустой" экземпляр нашей модели ManualLinearRegression
# и отправляет её на устройство
dummy = ManualLinearRegression().to(device)

### Прямой проход

In [ ]:
# Устанавливает скорость обучения - это "eta" — греческая буква похожая
# на n
lr = 0.1

# Шаг 0 - Случайным образом инициализирует параметры "b" и "w"
torch.manual_seed(42)
# Теперь мы можем создать модель и сразу отправить её на устройство
model = ManualLinearRegression().to(device)

# Определяет оптимизатор вида SGD для обновления параметров
# (теперь получаемых непосредственно из модели)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Определяет функцию потерь как MSE
loss_fn = nn.MSELoss(reduction='mean')

# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    model.train() # Что это?!?

    # Шаг 1 - Вычисляет предсказанные выходные значения модели -  прямой проход
    # Больше никаких ручных предсказаний!
    yhat = model(x_train_tensor)
    
    # Шаг 2 - Вычисляет потери
    loss = loss_fn(yhat, y_train_tensor)

    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    loss.backward()
    
    # Шаг 4 - Обновляет параметры, используя градиенты и скорость обучения
    optimizer.step()
    optimizer.zero_grad()
    
# Мы также можем проверить параметры модели, используя метод state_dict
print(model.state_dict())

### train

In [ ]:
# train() включает режим обучения, но не выполняет шаг оптимизации.
# Для Linear режим не меняет формулу; он важен, например, для Dropout и BatchNorm.

### Вложенные модели

In [ ]:
linear = nn.Linear(1, 1)
linear

In [ ]:
linear.state_dict()

In [ ]:
class MyLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        # Вместо наших пользовательских параметров, мы используем модель Linear
        # с единственным входом и единственным выходом
        self.linear = nn.Linear(1, 1)
                
    def forward(self, x):
        # Теперь необходим только вызов
        return self.linear(x)

In [ ]:
torch.manual_seed(42)
dummy = MyLinearRegression().to(device)
list(dummy.parameters())

In [ ]:
dummy.state_dict()

### Последовательные модели

In [ ]:
torch.manual_seed(42)
# В качестве альтернативы можно использовать модель Sequential
model = nn.Sequential(nn.Linear(1, 1)).to(device)

model.state_dict()

### Слои

Два линейных слоя без нелинейной активации остаются одним аффинным
преобразованием. Здесь сравниваем организацию модели, а не добавляем нелинейность.

In [ ]:
torch.manual_seed(42)
# Последовательность двух линейных слоёв
model = nn.Sequential(nn.Linear(3, 5), nn.Linear(5, 1)).to(device)

model.state_dict()

In [ ]:
torch.manual_seed(42)
# Последовательность двух линейных слоёв
model = nn.Sequential()
model.add_module('layer1', nn.Linear(3, 5))
model.add_module('layer2', nn.Linear(5, 1))
model.to(device)

<a id="section-10"></a>

## 10. Собираем всё вместе


### Подготовка данных

#### Подготовка данных V0

In [ ]:
%%writefile {lesson_dir}/data_preparation/v0.py

device = select_device()

# Наши данные были массивами Numpy, но нам нужно преобразовать их
# в тензоры PyTorch, а затем отправить их на
# выбранное устройство
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

In [ ]:
%run -i {lesson_dir}/data_preparation/v0.py

### Конфигурация модели

#### Конфигурация модели V0

In [ ]:
%%writefile {lesson_dir}/model_configuration/v0.py

# Сейчас эта часть лишняя, но не будет лишней, когда мы
# введём понятие Datasets...
device = select_device()

# Устанавливает скорость обучения — это "эта", греческая буква похожая на n
lr = 0.1

torch.manual_seed(42)
# Теперь мы можем создать модель и сразу отправить её на устройство
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Определяет оптимизатор вида SGD для обновления параметров
# (теперь получаемых непосредственно из модели)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Определяет функцию потерь как MSE
loss_fn = nn.MSELoss(reduction='mean')

In [ ]:
%run -i {lesson_dir}/model_configuration/v0.py

### Обучение модели

#### Обучение модели V0

In [ ]:
%%writefile {lesson_dir}/model_training/v0.py

# Определяет количество эпох
n_epochs = 1000

for epoch in range(n_epochs):
    # Устанавливает модель в режим ОБУЧЕНИЯ
    model.train()

    # Шаг 1 - Вычисляет предсказанные выходные значения модели - прямой проход
    yhat = model(x_train_tensor)
    
    # Шаг 2 - Вычисляет потери
    loss = loss_fn(yhat, y_train_tensor)

    # Шаг 3 - Вычисляет градиенты для обоих параметров "b" и "w"
    loss.backward()
    
    # Шаг 4 - Обновляет параметры, используя градиенты и
    # скорость обучения
    optimizer.step()
    optimizer.zero_grad()

In [ ]:
%run -i {lesson_dir}/model_training/v0.py

In [ ]:
print(model.state_dict())

### Дополнение курса: оценка после обучения

Сравним train и validation MSE при одних и тех же итоговых параметрах.
Валидационные данные не участвуют в backward. `eval()` переключает режим
модулей, а `no_grad()` отключает запись операций для вычисления градиентов.

In [ ]:
model.eval()
with torch.no_grad():
    train_mse = loss_fn(model(x_train_tensor), y_train_tensor).item()
    x_val_tensor = torch.as_tensor(x_val).float().to(device)
    y_val_tensor = torch.as_tensor(y_val).float().to(device)
    val_mse = loss_fn(model(x_val_tensor), y_val_tensor).item()
print(f'MSE на обучающей выборке: {train_mse:.8f}')
print(f'MSE на валидационной выборке: {val_mse:.8f}')

<a id="section-11"></a>

## 11. Задания

Шесть коротких упражнений на материал главы. Подготовка перед каждым заданием
независима от решений предыдущих. Для ручной сверки используем CPU и float64.

In [ ]:
main_device = device
device = torch.device('cpu')
dtype = torch.float64

### T1. Подготовить данные
Создайте `train_x` и `train_y` из `x_train` и `y_train` с явными `dtype=dtype`, `device=device`. Нужны независимые копии без градиентов, формы `(80, 1)`. Две строки.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">train_x = torch.tensor(x_train, dtype=dtype, device=device)
train_y = torch.tensor(y_train, dtype=dtype, device=device)</code></pre>

</details>

In [ ]:
raise NotImplementedError('T1: заполните пропуск')

In [ ]:
assert train_x.shape == train_y.shape == (80, 1)
assert train_x.dtype == train_y.dtype == dtype
assert train_x.device == train_y.device == device
assert not train_x.requires_grad and not train_y.requires_grad
np.testing.assert_allclose(train_x.numpy(), x_train)
np.testing.assert_allclose(train_y.numpy(), y_train)
assert not np.shares_memory(train_x.numpy(), x_train)
assert not np.shares_memory(train_y.numpy(), y_train)
print('T1: проверка пройдена')

### T2. Одна ошибка на объект
Преобразуйте `targets` в столбец и вычислите `residuals`. Ожидается форма `(3, 1)`. Одна строка. Значения данных менять нельзя.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">residuals = predictions - targets.reshape(-1, 1)</code></pre>

</details>

In [ ]:
predictions = torch.tensor([[1.], [3.], [3.]], dtype=dtype)
targets = torch.tensor([1., 2., 4.], dtype=dtype)
wrong_residuals = predictions - targets
print('Форма:', tuple(wrong_residuals.shape))
print(wrong_residuals)
seminar_plots.plot_shapes();

In [ ]:
raise NotImplementedError('T2: заполните пропуск')

In [ ]:
assert residuals.shape == (3, 1)
torch.testing.assert_close(residuals, torch.tensor([[0.], [1.], [-1.]], dtype=dtype))
print('T2: проверка пройдена')

### T3. Вычислить градиенты
Вызовите обратный проход от `loss`. Одна строка. Ожидается `parameters.grad` формы `(2,)` в порядке `(b, w)`.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">loss.backward()</code></pre>

</details>

In [ ]:
parameters = torch.tensor([0., 1.], dtype=dtype, requires_grad=True)
sample_x, sample_y = torch.tensor(2., dtype=dtype), torch.tensor(3., dtype=dtype)
prediction = parameters[0] + parameters[1]*sample_x
loss = (prediction - sample_y).square()
prediction.retain_grad()
print('Предсказание:', prediction.item(), '| Потеря:', loss.item())
print('Градиент параметров до backward:', parameters.grad)
print('Лист / промежуточный узел:', parameters.is_leaf, prediction.is_leaf)

In [ ]:
raise NotImplementedError('T3: заполните пропуск')

In [ ]:
torch.testing.assert_close(parameters.grad, torch.tensor([-2., -4.], dtype=dtype))
torch.testing.assert_close(parameters.detach(), torch.tensor([0., 1.], dtype=dtype))
print('По параметрам:', parameters.grad)
print('По предсказанию:', prediction.grad)
print('По данным:', sample_x.grad, sample_y.grad)

### T4. Градиент только нового прохода
Очистите `p_acc.grad`, присвоив `None`, затем сделайте новый forward/backward для того же примера. Две строки. Ожидается градиент одного прохода.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">p_acc.grad = None
(p_acc[0] + 2*p_acc[1] - 3).square().backward()</code></pre>

</details>

In [ ]:
p_acc = torch.tensor([0., 1.], dtype=dtype, requires_grad=True)
for _ in range(2):
    current_loss = (p_acc[0] + 2*p_acc[1] - 3).square()
    current_loss.backward()
print('После двух новых графов:', p_acc.grad)
torch.testing.assert_close(p_acc.grad, torch.tensor([-4., -8.], dtype=dtype))

In [ ]:
raise NotImplementedError('T4: заполните пропуск')

In [ ]:
torch.testing.assert_close(p_acc.grad, torch.tensor([-2., -4.], dtype=dtype))
print('T4: проверка пройдена')

### T5. Один шаг
Внутри готового контекста измените `p_step` на месте по градиенту и `learning_rate`. Одна строка. Объект должен остаться листом с `requires_grad=True`.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">with torch.no_grad():
    p_step -= learning_rate*p_step.grad</code></pre>

</details>

In [ ]:
p_step = torch.tensor([0., 1.], dtype=dtype, requires_grad=True)
loss_before = (p_step[0] + 2*p_step[1] - 3).square()
loss_before.backward()
learning_rate = 0.1

In [ ]:
with torch.no_grad():
    raise NotImplementedError('T5: заполните пропуск')

In [ ]:
torch.testing.assert_close(p_step.detach(), torch.tensor([0.2, 1.4], dtype=dtype))
assert p_step.is_leaf and p_step.requires_grad
with torch.no_grad():
    prediction_after = p_step[0] + 2*p_step[1]
    loss_after = (prediction_after - 3).square()
print('Параметры:', p_step.detach(), '| Новая потеря:', loss_after.item())
assert loss_after.item() < 1e-20
print('Старый градиент ещё хранится:', p_step.grad)

### T6. Backward и шаг SGD
По `optimizer_loss` вычислите градиенты, затем выполните обновление через `optimizer`. Две строки. Ожидается тот же результат, что при ручном шаге.

<details>
<summary>Ответ</summary>

<pre><code class="language-python">optimizer_loss.backward()
optimizer.step()</code></pre>

</details>

In [ ]:
p_optimizer = torch.tensor([0., 1.], dtype=dtype, requires_grad=True)
optimizer = torch.optim.SGD([p_optimizer], lr=0.1)
optimizer.zero_grad(set_to_none=True)
optimizer_loss = (p_optimizer[0] + 2*p_optimizer[1] - 3).square()

In [ ]:
raise NotImplementedError('T6: заполните пропуск')

In [ ]:
torch.testing.assert_close(p_optimizer.detach(), torch.tensor([0.2, 1.4], dtype=dtype))
print('T6: SGD совпадает с ручным шагом')
optimizer.zero_grad(set_to_none=True)
assert p_optimizer.grad is None

## Материалы

Daniel Voigt Godoy, [Chapter 1](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter01.ipynb).
Основной код и рисунки адаптированы из PyTorchStepByStep, Copyright (c) 2020 Daniel Voigt Godoy,
[лицензия MIT](../../LICENSE-GODOY). Поддержка MPS, переносимый запуск, пояснения и T1–T6 добавлены для курса.

Следующий семинар: [Chapter 2](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter02.ipynb),
Dataset, DataLoader, мини-батчи и организация обучения.